# Task 14: Ensemble Methods - Bagging & Boosting
**PKCERT AI & Software Development Internship**

**Dataset**: Bank Marketing (UCI) - Predicting term deposit subscription.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# For Boosting
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries imported successfully!")

Libraries imported successfully!


In [4]:
df = pd.read_csv('../data/raw/bank-additional-full.csv', sep=';')

print("Dataset loaded!")
print("Shape:", df.shape)
print("\nTarget distribution (y - term deposit):")
print(df['y'].value_counts(normalize=True) * 100)

Dataset loaded!
Shape: (41188, 21)

Target distribution (y - term deposit):
y
no     88.734583
yes    11.265417
Name: proportion, dtype: float64


### Data Preprocessing

We will handle categorical encoding and scaling using a Pipeline for cleanliness and to prevent data leakage.

In [5]:
# Basic cleaning
df_clean = df.copy()

# Target to binary
df_clean['y'] = df_clean['y'].map({'yes': 1, 'no': 0})

X = df_clean.drop('y', axis=1)
y = df_clean['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (32950, 20)
Test shape: (8238, 20)


### Preprocessing Pipeline

To efficiently handle both numerical and categorical features, we use `ColumnTransformer`. This approach:
- Applies StandardScaler to numerical features
- Applies OneHotEncoder to categorical features
- Ensures consistent transformation between train and test sets

In [6]:
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_cols)
    ])

print("Categorical cols:", categorical_cols)
print("Numerical cols:", numerical_cols)

Categorical cols: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
Numerical cols: ['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']


## Part B: Bagging

Bagging (Bootstrap Aggregating) trains multiple models on different subsets of data and aggregates their predictions. We use Random Forest as a popular Bagging implementation.

In [7]:
from sklearn.ensemble import RandomForestClassifier

# Bagging Model (Random Forest)
bagging_model = RandomForestClassifier(n_estimators=100, random_state=42)
bagging_model.fit(preprocessor.fit_transform(X_train), y_train)  # Note: fit preprocessor here for simplicity

y_pred_bagging = bagging_model.predict(preprocessor.transform(X_test))

print("Bagging Model Performance:")
print(classification_report(y_test, y_pred_bagging))

Bagging Model Performance:
              precision    recall  f1-score   support

           0       0.94      0.97      0.95      7310
           1       0.67      0.49      0.57       928

    accuracy                           0.92      8238
   macro avg       0.81      0.73      0.76      8238
weighted avg       0.91      0.92      0.91      8238



### Part C: XGBoost Classifier

XGBoost is implemented as the Boosting model. It is known for high performance and efficiency in tabular data.

In [8]:
from xgboost import XGBClassifier

# XGBoost Model
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42, eval_metric='logloss')
xgb_model.fit(preprocessor.fit_transform(X_train), y_train)

y_pred_xgb = xgb_model.predict(preprocessor.transform(X_test))

print("XGBoost Model Performance:")
print(classification_report(y_test, y_pred_xgb))

XGBoost Model Performance:
              precision    recall  f1-score   support

           0       0.95      0.97      0.96      7310
           1       0.69      0.57      0.63       928

    accuracy                           0.92      8238
   macro avg       0.82      0.77      0.79      8238
weighted avg       0.92      0.92      0.92      8238



## Part D: Comparative Analysis

Bagging (Random Forest) and Boosting (XGBoost) both achieved strong performance. XGBoost showed slightly better results on the minority class.

In [9]:
print("Summary Comparison:")
print("Bagging Accuracy : 0.92")
print("XGBoost Accuracy : 0.92")
print("XGBoost is recommended for this dataset due to better handling of imbalance and speed.")

Summary Comparison:
Bagging Accuracy : 0.92
XGBoost Accuracy : 0.92
XGBoost is recommended for this dataset due to better handling of imbalance and speed.


### Conclusion

XGBoost is recommended for its superior performance and efficiency. It is widely used in industry for tabular data.